# Ноутбук 01: Train/Validation, обобщение и выбор feature set (без JSON)
Использует `feature_ranking.csv` из ЛР01 для определения candidate feature set.

In [1]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from lab_utils import *
%matplotlib inline
sns.set_style('whitegrid')

In [2]:
# Загрузка данных и разделение
datasets = load_course_datasets()
splits = {}
for name, df in datasets.items():
    X, y = split_xy(df)
    X_train, X_valid, X_test, y_train, y_valid, y_test = train_valid_test_split_stratified(
        X, y, test_size=0.2, valid_size=0.2, random_state=SEED
    )
    splits[name] = (X_train, X_valid, X_test, y_train, y_valid, y_test)
    print(f"{name}: train {X_train.shape}, valid {X_valid.shape}, test {X_test.shape}")

medical: train (9, 12), valid (3, 12), test (3, 12)
finance: train (9, 11), valid (3, 11), test (3, 11)


In [3]:
# Препроцессинг
transformed = {}
for name, (X_train, X_valid, X_test, y_train, y_valid, y_test) in splits.items():
    preprocessor = build_preprocessor(X_train)
    X_train_t, X_valid_t, X_test_t, feat_names = transform_with_names(
        preprocessor, X_train, X_valid, X_test
    )
    transformed[name] = (X_train_t, X_valid_t, X_test_t, feat_names, y_train, y_valid, y_test)

In [4]:
# Загружаем feature ranking из ЛР01
ranking_path = OUTPUT_DIR / 'feature_ranking.csv'
if not ranking_path.exists():
    import shutil
    src = Path('..') / '01-feature-importance-and-selection' / 'outputs' / 'feature_ranking.csv'
    if src.exists():
        shutil.copy(src, ranking_path)
        print(f"Скопирован {src}")
ranking_df = pd.read_csv(ranking_path)
print("Feature ranking loaded")

Feature ranking loaded


In [5]:
# Функция оценки одного feature set
def evaluate_feature_set(dataset_name, feature_set_name, model, ranking_df, transformed):
    X_train_t, X_valid_t, _, feat_names, y_train, y_valid, _ = transformed[dataset_name]
    if feature_set_name != 'full':
        selected = get_feature_set_features(dataset_name, feature_set_name, feat_names, ranking_df)
        X_train_sel = select_features(X_train_t, feat_names, selected)
        X_valid_sel = select_features(X_valid_t, feat_names, selected)
    else:
        X_train_sel, X_valid_sel = X_train_t, X_valid_t
    model_clone = model.__class__(**model.get_params())
    model_clone.set_params(random_state=SEED)
    _, fit_time, train_metrics, valid_metrics = measure_fit_and_split_metrics(
        model_clone, X_train_sel, y_train, X_valid_sel, y_valid
    )
    return (
        {'dataset': dataset_name, 'feature_set': feature_set_name, 'model': model.__class__.__name__,
         'split': 'train', 'accuracy': train_metrics['accuracy'], 'f1': train_metrics['f1'],
         'roc_auc': train_metrics['roc_auc'], 'fit_time_sec': fit_time},
        {'dataset': dataset_name, 'feature_set': feature_set_name, 'model': model.__class__.__name__,
         'split': 'validation', 'accuracy': valid_metrics['accuracy'], 'f1': valid_metrics['f1'],
         'roc_auc': valid_metrics['roc_auc'], 'fit_time_sec': fit_time}
    )

In [ ]:
# Прогон всех комбинаций
rows = []
models = make_default_models()
for ds_name in ['medical', 'finance']:
    candidate_sets = list_feature_set_names_from_ranking(ds_name, ranking_df)
    print(f"{ds_name}: {candidate_sets}")
    for model_name, model in models.items():
        for fs in candidate_sets:
            train_row, val_row = evaluate_feature_set(ds_name, fs, model, ranking_df, transformed)
            rows.append(train_row)
            rows.append(val_row)

audit_df = pd.DataFrame(rows)
audit_df.to_csv(OUTPUT_DIR / 'generalization_audit.csv', index=False)
print("Saved generalization_audit.csv")
audit_df.head()

medical: ['full', 'robust_D']


In [ ]:
# Выбор feature set для каждой модели
decisions_df = build_model_feature_set_decisions(audit_df)
decisions_df.to_csv(OUTPUT_DIR / 'model_feature_set_decisions.csv', index=False)
print("Saved model_feature_set_decisions.csv")
decisions_df

In [ ]:
# Дополнительная сводка по обобщению
summary = build_generalization_selection_summary(audit_df)
summary.to_csv(OUTPUT_DIR / 'validation_curve_results.csv', index=False)  # заглушка, можно сохранить как есть
print("Saved validation_curve_results.csv (summary placeholder)")
summary.head()

## Narrative
- **Переобучение**: ... (добавьте свои выводы)
- **Выбор feature set**: ...
- **Выводы**: ...